# Audio Datasets Creation System — Main Kaggle Runner

End-to-end Kaggle runner integrating **Student 1 (Steps 1–4)** and **Student 2 (Steps 5–6)** in a single session with **zero manual download/upload between phases**.

### Kaggle Session Settings (Right Panel)
- **Accelerator:** GPU (`T4` or `P100`)
- **Internet:** **ON** (needed for `git clone`, `pip`, `yt-dlp`, AST, Silero VAD, Whisper large-v3, and Audar-ASR-V1-Turbo)

### Integrated Pipeline Flow
1. **Clone & Setup:** Clones `Audio-Datasets-Creation-System` into `/kaggle/working/Audio-Datasets-Creation-System`. Everything is saved to and loaded from `/kaggle/working/corpus_data/<video_id>/`.
2. **Phase 1 — Student 1 (Steps 1–4):** Download / ingest media → 16 kHz mono 16-bit WAV → VAD pauses & Otsu silence floor → AST 4-class labelling (`speech`, `music`, `mixed`, `silence`) → Cut Engine v2 DP segmentation (5–15 s) → 10-gate QC & `segments_student1.csv` backup.
3. **Phase 2 — Student 2 Step 5 (SNR):** Passes **ALL segments** (`speech`, `mixed`, `music`, `silence`) to `speech_corpus.snr.compute_snr_for_segments`, computing both Energy SNR and WADA-SNR (`clean` vs `noisy`) and saving `segments_snr.csv` + `snr_histogram.png` + `segment_length_histogram.png`.
4. **Phase 3 — Student 2 Step 6 (Dual ASR):** Filters strictly to **`speech` and `mixed` segments only** (`music` and `silence` bypass ASR) → `openai/whisper-large-v3` (saved & GPU freed) → `audarai/Audar-ASR-V1-Turbo` (saved & GPU freed) → Arabic normalization + `jiwer` CER/WER agreement (`accepted`, `human_choice`, `difficult`) → merges back with non-ASR rows into Annotation-App-ready `segments.csv` and `segments_full.csv`.
5. **Phase 4 — Step 7 Deliverables & Packaging:** Per-video summary table (`corpus_summary_step7.csv`), `.wav` + `.txt` pairs, global corpus manifest, zero-leakage splits (when >= 3 videos), and `/kaggle/working/corpus_final_output.zip`.

In [ ]:
# Cell 1: Clone Repository on Kaggle & Install Dependencies
import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_ROOT = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
REPO_URL = "https://github.com/CaptainLevi04/Audio-Datasets-Creation-System.git"
UPDATE_REPO = True  # Set False to keep local edits if re-running

if (WORK_ROOT / "speech_corpus").is_dir():
    REPO_DIR = WORK_ROOT
elif (Path.cwd() / "speech_corpus").is_dir():
    REPO_DIR = Path.cwd()
else:
    REPO_DIR = WORK_ROOT / "Audio-Datasets-Creation-System"
    if not REPO_DIR.exists():
        print(f"Cloning repository into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    elif UPDATE_REPO and (REPO_DIR / ".git").exists():
        print(f"Updating repository in {REPO_DIR} ...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

# Root working directory at WORK_ROOT so saved paths stay relative and portable
os.chdir(WORK_ROOT)
if str(REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIR.resolve()))

# Ensure ffmpeg is available
if shutil.which("ffmpeg") is None:
    print("ffmpeg not found on PATH; attempting installation...")
    if sys.platform.startswith("linux"):
        subprocess.run("apt-get update -qq && apt-get install -y -qq ffmpeg", shell=True, check=False)
    if shutil.which("ffmpeg") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "imageio-ffmpeg"])

# Install Student 1 + Student 2 dependencies
# Note: Do NOT install NeMo here (conflicts with qwen-asr on Kaggle).
!pip install -q "yt-dlp[default]" qwen-asr librosa soundfile jiwer faster-whisper tqdm ipywidgets

import torch
import numpy as np
import pandas as pd

# Verify modular imports from Student 1 and Student 2
from speech_corpus.audio import DEFAULT_SR, read_wav, write_wav
from speech_corpus.downloader import (
    download_video, extract_video_id, find_cached_video, ingest_local_file
)
from speech_corpus import ast_model
from speech_corpus import segmentation
from speech_corpus.ast_model import classify_audio, validate_cached_regions, save_regions
from speech_corpus.segmentation import detect_speech_activity_overlay
from speech_corpus.cut_engine_v2 import cut_speech_and_mixed_segments
from speech_corpus.visualization import plot_spectrogram_regions, plot_segments_overview
from speech_corpus.quality_control import (
    validate_segments_integrity, generate_qc_markdown_report,
    video_summary, split_corpus
)
from speech_corpus.snr import (
    DEFAULT_THRESHOLD_DB, compute_snr_for_segments,
    estimate_segment_snr, plot_snr_histogram
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dev_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Environment ready | ON_KAGGLE={ON_KAGGLE} | WORK_ROOT={WORK_ROOT}")
print(f"Repo: {REPO_DIR} | Device: {device.upper()} ({dev_name})")

## Configuration
Edit only the cell below. You can pass 1 or more videos (e.g., all 3 videos for Step 7) via YouTube URLs or local files (including `/kaggle/input/...` media files). All intermediate and final artifacts are saved under `/kaggle/working/corpus_data/` and loaded automatically by subsequent phases.

In [ ]:
# Cell 2: Pipeline Configuration
# Each entry in SOURCES can be:
#   - a YouTube URL string: "https://www.youtube.com/watch?v=..."
#   - a dict with url:      {"url": "https://www.youtube.com/watch?v=...", "video_id": "video1_lecture"}
#   - a dict with file:     {"file": "/kaggle/input/my-videos/lecture.mp4", "video_id": "video1_lecture"}
SOURCES = [
    {"url": "https://www.youtube.com/watch?v=gK017cCH36k"},
]

USE_MOCK_DEMO = False          # True = run offline synthetic smoke test (ignores SOURCES)
COOKIES_FILE = None            # Optional path to YouTube cookies.txt (e.g. "/kaggle/input/cookies/cookies.txt")
ATTACHED_OUTPUT_ROOT = None    # Optional: Path("/kaggle/input/<prev-dataset>/corpus_data") to copy prior cached outputs into working dir

DATA_ROOT = WORK_ROOT / "corpus_data"  # Saved in /kaggle/working/corpus_data on Kaggle
DATA_ROOT.mkdir(parents=True, exist_ok=True)
ANNOTATOR = "Team"

# --- Phase 1: Student 1 (Steps 1-4) ---
SILENCE_THRESHOLD_S = 0.25
FORCE_RECLASSIFY = False
FORCE_RECUT = False
USE_WORD_VERIFICATION = False  # True enables faster-whisper Arabic word boundary check during Step 4
WORD_MODEL_SIZE = "small"

# --- Phase 2: Student 2 Step 5 (SNR on ALL segments) ---
SNR_THRESHOLD_DB = 25.0        # speech_corpus.snr default is 25.0 dB (PDF starts at 20 dB)
SNR_COMBINE_MODE = "mean"      # "mean", "wada", "energy", or "min"
FORCE_RESNR = False

# --- Phase 3: Student 2 Step 6 (Dual ASR on SPEECH + MIXED segments only) ---
ASR_LABELS = ("speech", "mixed")
ASR_LIMIT = None               # e.g., 10 for quick test; None = all speech + mixed segments
FORCE_RETRANSCRIBE = False

WHISPER_ID = "openai/whisper-large-v3"
WHISPER_BATCH = 4              # Lower to 2 or 1 on CUDA OOM
WHISPER_BEAMS = 5

AUDAR_ID = "audarai/Audar-ASR-V1-Turbo"
AUDAR_LANG = "Arabic"
AUDAR_BATCH = 8                # Lower to 4 on CUDA OOM

SR = DEFAULT_SR                # 16000 Hz

# Optional: hydrate DATA_ROOT from an attached previous Kaggle dataset
if ATTACHED_OUTPUT_ROOT is not None:
    src_root = Path(ATTACHED_OUTPUT_ROOT)
    if src_root.exists():
        for item in src_root.iterdir():
            dst = DATA_ROOT / item.name
            if item.is_dir() and not dst.exists():
                shutil.copytree(item, dst)
        print(f"Copied cached outputs from {src_root} -> {DATA_ROOT.resolve()}")

print(f"Configured {len(SOURCES)} source(s) | USE_MOCK_DEMO={USE_MOCK_DEMO} | DATA_ROOT={DATA_ROOT.resolve()}")

## Phase 1 — Student 1 Pipeline (Steps 1–4: Download, 16 kHz Audio, AST Labelling, Cut Engine v2, 10-Gate QC)
Saves `audio.wav`, `info.json`, `regions.csv`, `segments/*.wav`, `qc_report.md`, `video_summary.csv`, and preserves Student 1's full 37-column manifest in `segments_student1.csv` inside `/kaggle/working/corpus_data/<video_id>/`.

In [ ]:
# Cell 3: Run Student 1 Pipeline (Steps 1-4)
import json
import matplotlib.pyplot as plt
from IPython.display import display

def _resolve_local_file(file_str: str) -> Path:
    p = Path(file_str)
    if p.exists():
        return p
    if Path("/kaggle/input").exists():
        matches = list(Path("/kaggle/input").rglob(p.name))
        if matches:
            print(f"Resolved local file from Kaggle input: {matches[0]}")
            return matches[0]
    raise FileNotFoundError(f"Could not find media file: {file_str}")

def _prepare_jobs():
    if USE_MOCK_DEMO:
        return [{"mode": "mock", "video_id": "mock_demo_audio"}]
    jobs = []
    for idx, entry in enumerate(SOURCES, 1):
        if isinstance(entry, str):
            entry = {"url": entry}
        if entry.get("file"):
            fp = _resolve_local_file(entry["file"])
            vid = entry.get("video_id") or extract_video_id(str(fp))
            jobs.append({"mode": "file", "file": fp, "video_id": vid})
        elif entry.get("url"):
            url = entry["url"]
            vid = entry.get("video_id") or extract_video_id(url)
            jobs.append({"mode": "url", "url": url, "video_id": vid})
        else:
            raise ValueError(f"Invalid SOURCES entry #{idx}: {entry}")
    return jobs

jobs = _prepare_jobs()
video_dirs = []

for j_idx, job in enumerate(jobs, 1):
    vid = job["video_id"]
    video_dir = DATA_ROOT / vid
    video_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 72)
    print(f"[Phase 1 | Video {j_idx}/{len(jobs)}] video_id={vid} -> {video_dir}")
    print("=" * 72)

    # --- Steps 1 & 2: Ingest & 16 kHz Mono WAV ---
    audio_path = video_dir / "audio.wav"
    info_path = video_dir / "info.json"
    if audio_path.exists() and info_path.exists():
        print(f"Found cached audio.wav & info.json in {video_dir}")
        info = json.loads(info_path.read_text(encoding="utf-8"))
        audio_samples, sr = read_wav(audio_path, target_sr=SR)
    elif job["mode"] == "mock":
        from speech_corpus.labelling import build_mock_audio
        audio_samples, _ = build_mock_audio(seed=42, sr=SR)
        write_wav(audio_path, audio_samples, sample_rate=SR)
        sr = SR
        info = {
            "video_id": vid, "source": "mock", "url": None,
            "title": "Synthetic Audio Demo", "channel": "Synthesizer",
            "duration_s": round(len(audio_samples) / SR, 2),
            "license": "Creative Commons Attribution", "is_creative_commons": True,
        }
        info_path.write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
    elif job["mode"] == "file":
        audio_path, info = ingest_local_file(job["file"], video_dir)
        audio_samples, sr = read_wav(audio_path, target_sr=SR)
    else:
        cached = find_cached_video(job["url"], search_dirs=[DATA_ROOT])
        if cached is not None:
            audio_path, info = cached
            video_dir = audio_path.parent
            audio_samples, sr = read_wav(audio_path, target_sr=SR)
        else:
            audio_path, info = download_video(job["url"], out_dir=video_dir, cookies_file=COOKIES_FILE)
            audio_samples, sr = read_wav(audio_path, target_sr=SR)

    video_dirs.append(video_dir)
    dur_s = len(audio_samples) / SR
    print(f"Audio ready: {dur_s:.1f}s ({dur_s/60:.2f} min) @ {SR} Hz | License: {info.get('license')}")

    # --- Step 3.0 & Step 3: VAD Pauses + AST 4-Class Region Labelling ---
    s1_backup_csv = video_dir / "segments_student1.csv"
    regions_csv = video_dir / "regions.csv"
    windows_csv = video_dir / "windows_ast.csv"

    vad_timeline_df, pauses_df = detect_speech_activity_overlay(
        audio=audio_samples, sr=SR,
        silence_threshold_s=SILENCE_THRESHOLD_S,
        min_pause_ms=120, vad_backend="auto",
    )

    if (not FORCE_RECLASSIFY) and validate_cached_regions(
        regions_csv, expected_source_id=video_dir.name, expected_duration=dur_s, expected_version="2.2"
    ):
        print(f"Loaded cached regions (v2.2): {regions_csv}")
        regions_df = pd.read_csv(regions_csv)
    else:
        print("Running AST macro-region classification...")
        regions_df, windows_df = classify_audio(
            y=audio_samples, sr=SR, win_s=10.0, hop_s=5.0, batch_size=32, pauses_df=pauses_df
        )
        save_regions(regions_df, regions_csv, source_id=video_dir.name, duration=dur_s, version="2.2")
        if windows_df is not None and not windows_df.empty:
            windows_df.to_csv(windows_csv, index=False)

    full_spec_path = video_dir / "full_spectrogram.png"
    if not full_spec_path.exists() or FORCE_RECLASSIFY:
        fig = plot_spectrogram_regions(
            audio_samples, regions_df, sr=SR,
            title=f"Full Audio Spectrogram - {str(info.get('title', vid))[:40]}",
            save_path=full_spec_path,
        )
        plt.close(fig)

    # --- Step 4: Cut Engine v2 (5-15s DP across ALL 4 labels) ---
    if (not FORCE_RECUT) and s1_backup_csv.exists() and (video_dir / "segments").exists():
        segments_df = pd.read_csv(s1_backup_csv, dtype={"segment_id": str})
        print(f"Loaded cached Student 1 segments ({len(segments_df)} rows): {s1_backup_csv}")
    else:
        segments_df = cut_speech_and_mixed_segments(
            audio=audio_samples,
            regions_df=regions_df,
            out_dir=video_dir,
            sr=SR,
            pauses=pauses_df,
            use_word_verification=USE_WORD_VERIFICATION,
            word_model_size=WORD_MODEL_SIZE,
            run_ablation=True,
        )
        # Preserve full 37-column Student 1 manifest BEFORE Step 5 overwrites segments.csv
        segments_df.to_csv(s1_backup_csv, index=False)

    if not (video_dir / "segments").exists() and (video_dir / "segments_staging").exists():
        print(f"WARNING: Cut Engine v2 QC failed for {vid}; segments remained in segments_staging/.")

    overview_path = video_dir / "segments_overview.png"
    if not overview_path.exists() or FORCE_RECUT:
        fig = plot_segments_overview(audio_samples, segments_df, sr=SR, max_segments=20, save_path=overview_path)
        plt.close(fig)

    # --- Step 4.5: 10-Gate QC & Video Summary (must run on Student 1 manifest) ---
    qc_results = validate_segments_integrity(segments_df, video_dir=video_dir, total_dur=dur_s, sr=SR)
    generate_qc_markdown_report(segments_df, report_path=video_dir / "qc_report.md", video_dir=video_dir, qc=qc_results)
    vsum_df = video_summary(regions_df=regions_df, segments_df=segments_df, info=info, video_dir=video_dir)

    lbl_counts = segments_df["label"].astype(str).str.lower().value_counts().to_dict()
    print(f"QC Status: {qc_results['status']} ({qc_results['summary']['passed_gates']}/{qc_results['summary']['total_gates']} gates)")
    print(f"Student 1 Segments: {len(segments_df)} total | By label: {lbl_counts}")
    display(vsum_df)

    # Free audio array from RAM before next video / ASR
    del audio_samples
    gc.collect()

# Free AST & Silero GPU caches before loading Whisper large-v3 / Audar in Phase 3
ast_model._AST_CACHE.clear()
segmentation._SILERO.clear()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nPhase 1 complete. All Student 1 outputs saved under:", DATA_ROOT.resolve())

## Phase 2 — Student 2 Step 5: SNR Estimation on ALL Segments
Loads `segments_student1.csv` and `segments/*.wav` directly from `/kaggle/working/corpus_data/<video_id>/`. Passes **all segments** (`speech`, `mixed`, `music`, `silence`) to `speech_corpus.snr.compute_snr_for_segments`, writes `segments_snr.csv` + `snr_diagnostics.csv` (Energy SNR & WADA-SNR), and saves `snr_histogram.png` and `segment_length_histogram.png`.

In [ ]:
# Cell 4: Run Student 2 Step 5 (SNR on ALL Segments)
import matplotlib.pyplot as plt
from IPython.display import display

# Auto-discover video folders directly on Kaggle disk if cell is run standalone
discovered_dirs = sorted([
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and ((p / "segments_student1.csv").exists() or (p / "segments.csv").exists())
])
assert discovered_dirs, f"No processed videos found in {DATA_ROOT.resolve()}. Run Phase 1 first."

for video_dir in discovered_dirs:
    print("=" * 72)
    print(f"[Phase 2 | Step 5 SNR] {video_dir}")
    print("=" * 72)

    s1_csv = video_dir / "segments_student1.csv"
    src_csv = s1_csv if s1_csv.exists() else (video_dir / "segments.csv")
    seg_src_df = pd.read_csv(src_csv, dtype={"segment_id": str})

    # Guard against re-running on an already-SNR'd CSV whose audio_file includes DATA_ROOT prefix
    if "wav_file" not in seg_src_df.columns and "audio_file" in seg_src_df.columns:
        seg_src_df["wav_file"] = seg_src_df["audio_file"].map(
            lambda x: f"segments/{Path(str(x)).name}" if pd.notnull(x) and not (video_dir / str(x)).exists() else x
        )

    # Verify ALL segments (speech, mixed, music, silence) are passed to SNR
    lbl_counts = seg_src_df["label"].astype(str).str.lower().value_counts().to_dict()
    print(f"Passing ALL {len(seg_src_df)} segments to SNR: {lbl_counts}")

    snr_ckpt = video_dir / "segments_snr.csv"
    diag_csv = video_dir / "snr_diagnostics.csv"

    if (not FORCE_RESNR) and snr_ckpt.exists() and diag_csv.exists():
        snr_df = pd.read_csv(snr_ckpt, dtype={"segment_id": str})
        print(f"Loaded cached SNR results: {snr_ckpt}")
    else:
        # Student 2 compute_snr_for_segments writes segments.csv in 16-col review format
        snr_df, _ = compute_snr_for_segments(
            seg_src_df,
            base_dir=video_dir,
            threshold_db=SNR_THRESHOLD_DB,
            combine=SNR_COMBINE_MODE,
            save_csv=True,
            verbose=True,
        )
        snr_df.to_csv(snr_ckpt, index=False, encoding="utf-8-sig")

        # Save per-segment Energy & WADA breakdown for the project report
        diag_rows = []
        for _, row in seg_src_df.iterrows():
            rel_wav = row["wav_file"] if "wav_file" in seg_src_df.columns else row["audio_file"]
            wp = video_dir / Path(str(rel_wav))
            if not wp.exists():
                wp = video_dir / "segments" / Path(str(rel_wav)).name
            samples, sr_w = read_wav(wp, target_sr=SR)
            est = estimate_segment_snr(samples, sr=sr_w, threshold_db=SNR_THRESHOLD_DB, combine=SNR_COMBINE_MODE)
            diag_rows.append({
                "segment_id": row["segment_id"],
                "label": row["label"],
                "snr_energy_db": est["snr_energy_db"],
                "snr_wada_db": est["snr_wada_db"],
                "snr": est["snr"],
                "quality": est["quality"],
                "snr_disagree": est["snr_disagree"],
            })
        pd.DataFrame(diag_rows).to_csv(diag_csv, index=False, encoding="utf-8-sig")

    # Save SNR histogram & Segment-length histogram (Deliverables)
    snr_hist_path = video_dir / "snr_histogram.png"
    fig = plot_snr_histogram(snr_df, threshold_db=SNR_THRESHOLD_DB, save_path=snr_hist_path)
    plt.close(fig)

    len_hist_path = video_dir / "segment_length_histogram.png"
    durs_seg = pd.to_numeric(snr_df["end"], errors="coerce") - pd.to_numeric(snr_df["start"], errors="coerce")
    fig_len, ax_len = plt.subplots(figsize=(8, 4))
    ax_len.hist(durs_seg.dropna(), bins=25, color="#2ca02c", edgecolor="black", alpha=0.85)
    ax_len.axvline(5.0, color="red", ls="--", label="5s min")
    ax_len.axvline(15.0, color="red", ls="--", label="15s max")
    ax_len.set_xlabel("Segment duration (s)")
    ax_len.set_ylabel("Number of segments")
    ax_len.set_title(f"Segment-Length Distribution ({video_dir.name})")
    ax_len.legend()
    fig_len.tight_layout()
    fig_len.savefig(len_hist_path, dpi=200)
    plt.close(fig_len)

    q_counts = snr_df["quality"].value_counts().to_dict()
    print(f"Quality counts: {q_counts} | Mean SNR: {pd.to_numeric(snr_df['snr'], errors='coerce').mean():.2f} dB")
    display(snr_df[["segment_id", "start", "end", "label", "snr", "quality"]].head(8))

## Phase 3 — Student 2 Step 6: Dual ASR (`speech` & `mixed` Segments ONLY) + Agreement Rule
Loads `segments_snr.csv` directly from `/kaggle/working/corpus_data/<video_id>/`.
1. **Strict Filtering:** Splits segments so **ONLY `speech` and `mixed`** are passed to ASR (`music` and `silence` bypass ASR cleanly).
2. **Model A (`openai/whisper-large-v3`):** Transcribes all `speech` + `mixed` segments across all videos, saves `asr_whisper_ckpt.csv` per video, and frees GPU VRAM.
3. **Model B (`audarai/Audar-ASR-V1-Turbo`):** Transcribes all `speech` + `mixed` segments across all videos, saves `asr_audar_ckpt.csv` per video, and frees GPU VRAM.
4. **Comparison & Merge:** Applies Arabic normalization + `jiwer` CER/WER rule (`accepted`, `human_choice`, `difficult`), merges `speech`/`mixed` back with `music`/`silence` (`status="no_asr"`, `review_status="skipped"`), and writes final `segments.csv` (16-col Annotation App schema), `segments_full.csv`, and `transcripts/*.txt`.

In [ ]:
# Cell 5: Filter Speech & Mixed Segments Only + Run ASR Model A (Whisper large-v3)
from tqdm.auto import tqdm

def _resolve_segment_wav(video_dir: Path, raw_path: str) -> Path:
    p = Path(str(raw_path))
    candidates = [
        p if p.is_absolute() else None,
        WORK_ROOT / p,
        video_dir / p,
        video_dir / "segments" / p.name,
        video_dir / "segments_staging" / p.name,
    ]
    for c in candidates:
        if c is not None and c.exists():
            return c
    raise FileNotFoundError(f"Missing segment WAV '{raw_path}' for {video_dir}")

discovered_dirs = sorted([
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and (p / "segments_snr.csv").exists()
])
assert discovered_dirs, f"No segments_snr.csv found under {DATA_ROOT.resolve()}. Run Phase 2 first."

asr_jobs = {}
allowed_set = {s.lower() for s in ASR_LABELS}

for video_dir in discovered_dirs:
    snr_df = pd.read_csv(video_dir / "segments_snr.csv", dtype={"segment_id": str})
    snr_df["_row_idx"] = np.arange(len(snr_df))
    lbl_norm = snr_df["label"].astype(str).str.strip().str.lower()

    asr_mask = lbl_norm.isin(allowed_set)
    df_asr_all = snr_df[asr_mask].copy().reset_index(drop=True)
    df_non_asr = snr_df[~asr_mask].copy().reset_index(drop=True)

    if ASR_LIMIT is not None:
        df_asr = df_asr_all.head(ASR_LIMIT).copy().reset_index(drop=True)
        df_asr_skipped = df_asr_all.iloc[ASR_LIMIT:].copy().reset_index(drop=True)
    else:
        df_asr = df_asr_all
        df_asr_skipped = pd.DataFrame(columns=snr_df.columns)

    # Assertion: no music or silence segments ever enter ASR
    if not df_asr.empty:
        bad = set(df_asr["label"].astype(str).str.strip().str.lower()) - allowed_set
        assert not bad, f"Unexpected labels passed to ASR: {bad}"

    print(
        f"[{video_dir.name}] Total(SNR)={len(snr_df)} | "
        f"ASR(speech+mixed)={len(df_asr)} | "
        f"Non-ASR(music+silence)={len(df_non_asr)}"
        + (f" | Deferred(ASR_LIMIT)={len(df_asr_skipped)}" if len(df_asr_skipped) else "")
    )

    audio_list = []
    for row in df_asr.itertuples():
        wp = _resolve_segment_wav(video_dir, row.audio_file)
        x, _ = read_wav(wp, target_sr=SR)
        audio_list.append(x)

    durs = np.array([len(a) / SR for a in audio_list], dtype=float) if audio_list else np.array([])
    if len(durs):
        assert durs.max() <= 30.5, (
            f"[{video_dir.name}] Segment exceeds 30.5s (max={durs.max():.2f}s); ASR models support <= 30s."
        )
        print(f"  Loaded {len(audio_list)} speech/mixed clips ({durs.sum()/60:.1f} min, max={durs.max():.1f}s)")
    rms = np.array([float(np.sqrt((a ** 2).mean())) for a in audio_list], dtype=float) if audio_list else np.array([])

    asr_jobs[video_dir] = {
        "snr_df": snr_df,
        "df_asr": df_asr,
        "df_non_asr": df_non_asr,
        "df_asr_skipped": df_asr_skipped,
        "audio": audio_list,
        "rms": rms,
    }

# Check which videos need Whisper inference
need_whisper = []
for vdir, job in asr_jobs.items():
    df_asr = job["df_asr"]
    ckpt = vdir / "asr_whisper_ckpt.csv"
    if len(df_asr) == 0:
        df_asr["model_A_transcript"] = []
        continue
    if (not FORCE_RETRANSCRIBE) and ckpt.exists():
        c_df = pd.read_csv(ckpt, dtype={"segment_id": str})
        if len(c_df) == len(df_asr) and [str(x) for x in c_df["segment_id"]] == [str(x) for x in df_asr["segment_id"]]:
            df_asr["model_A_transcript"] = c_df["model_A_transcript"].fillna("").tolist()
            print(f"[{vdir.name}] Loaded cached Whisper checkpoint: {ckpt}")
            continue
    need_whisper.append(vdir)

if need_whisper:
    from transformers import AutoProcessor, WhisperForConditionalGeneration
    w_dev = "cuda" if torch.cuda.is_available() else "cpu"
    dt = torch.float16 if w_dev == "cuda" else torch.float32
    print(f"Loading {WHISPER_ID} on {w_dev} (dtype={dt})...")
    w_proc = AutoProcessor.from_pretrained(WHISPER_ID)
    w_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_ID, torch_dtype=dt).to(w_dev).eval()

    for vdir in need_whisper:
        job = asr_jobs[vdir]
        df_asr = job["df_asr"]
        audio_list = job["audio"]
        rms = job["rms"]
        whisper_txt = []
        print(f"Transcribing {vdir.name} ({len(audio_list)} segments) with Whisper large-v3...")
        for i in tqdm(range(0, len(audio_list), WHISPER_BATCH)):
            batch = audio_list[i:i + WHISPER_BATCH]
            feats = w_proc(batch, sampling_rate=SR, return_tensors="pt").input_features.to(w_dev, dtype=dt)
            with torch.inference_mode():
                ids = w_model.generate(
                    feats, language="ar", task="transcribe",
                    num_beams=WHISPER_BEAMS, max_new_tokens=300,
                )
            texts = w_proc.batch_decode(ids, skip_special_tokens=True)
            for k, t in enumerate(texts):
                whisper_txt.append("" if rms[i + k] < 1e-3 else t.strip())

        df_asr["model_A_transcript"] = whisper_txt
        df_asr[["_row_idx", "segment_id", "label", "model_A_transcript"]].to_csv(
            vdir / "asr_whisper_ckpt.csv", index=False, encoding="utf-8-sig"
        )
        print(f"Saved Whisper checkpoint: {vdir / 'asr_whisper_ckpt.csv'}")

    del w_model, w_proc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"Freed GPU after Whisper | Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("Whisper inference skipped (all videos cached or no speech/mixed segments).")

In [ ]:
# Cell 6: Run ASR Model B (Audar-ASR-V1-Turbo) on Speech & Mixed Segments Only
need_audar = []
for vdir, job in asr_jobs.items():
    df_asr = job["df_asr"]
    ckpt = vdir / "asr_audar_ckpt.csv"
    if len(df_asr) == 0:
        df_asr["model_B_transcript"] = []
        continue
    if (not FORCE_RETRANSCRIBE) and ckpt.exists():
        c_df = pd.read_csv(ckpt, dtype={"segment_id": str})
        if len(c_df) == len(df_asr) and [str(x) for x in c_df["segment_id"]] == [str(x) for x in df_asr["segment_id"]]:
            df_asr["model_B_transcript"] = c_df["model_B_transcript"].fillna("").tolist()
            df_asr["second_model"] = "audar"
            print(f"[{vdir.name}] Loaded cached Audar checkpoint: {ckpt}")
            continue
    need_audar.append(vdir)

if need_audar:
    from qwen_asr import Qwen3ASRModel

    if torch.cuda.is_available():
        major, _ = torch.cuda.get_device_capability(0)
        adt = torch.bfloat16 if major >= 8 else torch.float16
        dev_map = "cuda:0"
    else:
        adt = torch.float32
        dev_map = "cpu"

    print(f"Loading {AUDAR_ID} on {dev_map} (dtype={adt})...")
    a_model = Qwen3ASRModel.from_pretrained(
        AUDAR_ID, dtype=adt, device_map=dev_map,
        max_inference_batch_size=AUDAR_BATCH, max_new_tokens=256,
    )
    a_model.model.to(adt)

    for vdir in need_audar:
        job = asr_jobs[vdir]
        df_asr = job["df_asr"]
        audio_list = job["audio"]
        rms = job["rms"]
        audar_txt = []
        print(f"Transcribing {vdir.name} ({len(audio_list)} segments) with Audar-ASR-V1-Turbo...")
        for i in tqdm(range(0, len(audio_list), AUDAR_BATCH)):
            batch = [(a, SR) for a in audio_list[i:i + AUDAR_BATCH]]
            out = a_model.transcribe(audio=batch, language=[AUDAR_LANG] * len(batch))
            audar_txt.extend(r.text.strip() for r in out)

        df_asr["second_model"] = "audar"
        df_asr["model_B_transcript"] = [("" if r < 1e-3 else t) for t, r in zip(audar_txt, rms)]
        df_asr[["_row_idx", "segment_id", "label", "model_B_transcript"]].to_csv(
            vdir / "asr_audar_ckpt.csv", index=False, encoding="utf-8-sig"
        )
        print(f"Saved Audar checkpoint: {vdir / 'asr_audar_ckpt.csv'}")

    del a_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"Freed GPU after Audar | Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("Audar inference skipped (all videos cached or no speech/mixed segments).")

In [ ]:
# Cell 7: Arabic Normalization, jiwer Comparison & Merge Back with Music/Silence
import re
import jiwer
from IPython.display import display

_DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
_ALEF = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا"})

def normalize_ar(t) -> str:
    t = "" if t is None or (isinstance(t, float) and t != t) else str(t)
    t = _DIAC.sub("", t).replace("ـ", "").translate(_ALEF)
    t = t.replace("ة", "ه").replace("ى", "ي")
    t = re.sub(r"[^\w\s]", " ", t).replace("_", " ")
    return " ".join(t.split())

def compare_transcripts(a, b) -> tuple[str, float, int]:
    na, nb = normalize_ar(a), normalize_ar(b)
    if not na or not nb:
        return "difficult", 1.0, 999
    w = jiwer.process_words(na, nb)
    n_diff = int(w.substitutions + w.deletions + w.insertions)
    cer = float(jiwer.cer(na, nb))
    if cer <= 0.02:
        return "accepted", round(cer, 4), n_diff
    status = "human_choice" if n_diff <= 3 else "difficult"
    return status, round(cer, 4), n_diff

APP_COLS = [
    "start", "end", "label", "snr", "quality",
    "model_A_transcript", "model_B_transcript", "final_transcript",
    "status", "review_status", "automatic_status",
    "segment_id", "audio_file",
    "changed_by", "changed_at", "change_log",
]

for vdir, job in asr_jobs.items():
    df_asr = job["df_asr"].copy()
    df_non_asr = job["df_non_asr"].copy()
    df_skipped = job["df_asr_skipped"].copy()

    if len(df_asr):
        statuses, cers, diffs = [], [], []
        for a, b in zip(df_asr["model_A_transcript"], df_asr["model_B_transcript"]):
            st, cer_v, nd_v = compare_transcripts(a, b)
            statuses.append(st)
            cers.append(cer_v)
            diffs.append(nd_v)
        df_asr["automatic_status"] = statuses
        df_asr["status"] = statuses
        df_asr["review_status"] = ["accepted" if s == "accepted" else "pending" for s in statuses]
        df_asr["final_transcript"] = [
            a if s == "accepted" else ""
            for a, s in zip(df_asr["model_A_transcript"], statuses)
        ]
        df_asr["cer"] = cers
        df_asr["n_diff_words"] = diffs
        df_asr["second_model"] = "audar"
        df_asr["changed_by"] = ""
        df_asr["changed_at"] = ""
        df_asr["change_log"] = "[]"

    if len(df_non_asr):
        df_non_asr["model_A_transcript"] = ""
        df_non_asr["model_B_transcript"] = ""
        df_non_asr["final_transcript"] = ""
        df_non_asr["status"] = "no_asr"
        df_non_asr["automatic_status"] = "no_asr"
        df_non_asr["review_status"] = "skipped"
        df_non_asr["cer"] = np.nan
        df_non_asr["n_diff_words"] = np.nan
        df_non_asr["second_model"] = ""
        df_non_asr["changed_by"] = ""
        df_non_asr["changed_at"] = ""
        df_non_asr["change_log"] = "[]"

    if len(df_skipped):
        df_skipped["model_A_transcript"] = ""
        df_skipped["model_B_transcript"] = ""
        df_skipped["final_transcript"] = ""
        df_skipped["status"] = "not_run"
        df_skipped["automatic_status"] = "not_run"
        df_skipped["review_status"] = "pending"
        df_skipped["cer"] = np.nan
        df_skipped["n_diff_words"] = np.nan
        df_skipped["second_model"] = ""
        df_skipped["changed_by"] = ""
        df_skipped["changed_at"] = ""
        df_skipped["change_log"] = "[]"

    parts = [p for p in (df_asr, df_non_asr, df_skipped) if len(p) > 0]
    merged_df = pd.concat(parts, ignore_index=True).sort_values("_row_idx").reset_index(drop=True)

    # Save exact 16-column Annotation App CSV (segments.csv & segments_asr.csv)
    df_out = merged_df[[c for c in APP_COLS if c in merged_df.columns]].copy()
    df_out.to_csv(vdir / "segments.csv", index=False, encoding="utf-8-sig")
    df_out.to_csv(vdir / "segments_asr.csv", index=False, encoding="utf-8-sig")

    # Build rich segments_full.csv (Student 1 metadata + Step 5 Energy/WADA SNR + Step 6 ASR metrics)
    s1_path = vdir / "segments_student1.csv"
    diag_path = vdir / "snr_diagnostics.csv"
    full_df = merged_df.copy()
    if s1_path.exists():
        s1_df = pd.read_csv(s1_path, dtype={"segment_id": str}).reset_index(drop=True)
        if len(s1_df) == len(full_df):
            if "status" in s1_df.columns:
                full_df["qc_status"] = s1_df["status"].values
            extra_s1 = [c for c in s1_df.columns if c not in full_df.columns]
            for c in extra_s1:
                full_df[c] = s1_df[c].values
    if diag_path.exists():
        d_df = pd.read_csv(diag_path, dtype={"segment_id": str}).reset_index(drop=True)
        if len(d_df) == len(full_df):
            for c in ["snr_energy_db", "snr_wada_db", "snr_disagree"]:
                if c in d_df.columns:
                    full_df[c] = d_df[c].values
    full_df.drop(columns=["_row_idx"], errors="ignore").to_csv(
        vdir / "segments_full.csv", index=False, encoding="utf-8-sig"
    )

    # Write per-segment .txt transcript files alongside segments/*.wav (Deliverable 1)
    txt_dir = vdir / "transcripts"
    txt_dir.mkdir(parents=True, exist_ok=True)
    for r in df_asr.itertuples():
        sid = str(r.segment_id)
        txt_path = txt_dir / f"segment_{sid}.txt"
        if r.status == "accepted":
            txt_path.write_text(str(r.final_transcript), encoding="utf-8")
        else:
            txt_path.write_text(
                f"[{r.status}]\nModel A (Whisper): {r.model_A_transcript}\nModel B (Audar): {r.model_B_transcript}\n",
                encoding="utf-8",
            )

    print(f"\n[{vdir.name}] Final status breakdown: {df_out['status'].value_counts().to_dict()}")
    print(f"  Saved: {vdir / 'segments.csv'} (16-col app format)")
    print(f"  Saved: {vdir / 'segments_full.csv'} ({len(full_df.columns)-1} cols)")
    display(df_out[["segment_id", "label", "snr", "quality", "model_A_transcript", "model_B_transcript", "status"]].head(10))

## Phase 4 — Step 7 Summary Table, Multi-Video Corpus Manifest, Zero-Leakage Splits & Kaggle Output Zip
Compiles the PDF Step 7 summary table across all processed videos, builds the unified corpus manifest, generates leakage-safe `train`/`val`/`test` splits (once >= 3 videos exist), and creates `/kaggle/working/corpus_final_output.zip`.

In [ ]:
# Cell 8: Step 7 Summary Table, Manifest, Splits & Final Output Packaging
import json
import zipfile
from IPython.display import display

video_folders = sorted([
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and (p / "segments.csv").exists() and (p / "regions.csv").exists()
])
assert video_folders, "No completed video folders found."

summary_rows = []
app_parts = []
full_parts = []

for vdir in video_folders:
    info = json.loads((vdir / "info.json").read_text(encoding="utf-8")) if (vdir / "info.json").exists() else {}
    reg_df = pd.read_csv(vdir / "regions.csv")
    seg_app = pd.read_csv(vdir / "segments.csv", dtype={"segment_id": str})
    app_parts.append(seg_app.assign(video_id=vdir.name))

    full_path = vdir / "segments_full.csv"
    if full_path.exists():
        full_parts.append(pd.read_csv(full_path, dtype={"segment_id": str}))

    reg_dur = reg_df.assign(d=reg_df["end"] - reg_df["start"]).groupby("label")["d"].sum() / 60.0
    asr_sub = seg_app[seg_app["label"].astype(str).str.lower().isin(["speech", "mixed"])]
    n_asr = max(1, len(asr_sub))
    st_counts = asr_sub["status"].value_counts().to_dict() if len(asr_sub) else {}
    mean_snr = pd.to_numeric(seg_app["snr"], errors="coerce").mean()

    summary_rows.append({
        "video_id": vdir.name,
        "title": info.get("title", vdir.name),
        "duration_min": round(float(reg_df["end"].max()) / 60.0, 2) if len(reg_df) else 0.0,
        "speech_min": round(float(reg_dur.get("speech", 0.0)), 2),
        "music_min": round(float(reg_dur.get("music", 0.0)), 2),
        "mixed_min": round(float(reg_dur.get("mixed", 0.0)), 2),
        "silence_min": round(float(reg_dur.get("silence", 0.0)), 2),
        "total_segments": len(seg_app),
        "asr_segments": len(asr_sub),
        "mean_snr_db": round(float(mean_snr), 2) if pd.notnull(mean_snr) else None,
        "clean_pct": round(100.0 * float((seg_app["quality"] == "clean").mean()), 1) if len(seg_app) else 0.0,
        "accepted_pct": round(100.0 * st_counts.get("accepted", 0) / n_asr, 1) if len(asr_sub) else 0.0,
        "human_choice_pct": round(100.0 * st_counts.get("human_choice", 0) / n_asr, 1) if len(asr_sub) else 0.0,
        "difficult_pct": round(100.0 * st_counts.get("difficult", 0) / n_asr, 1) if len(asr_sub) else 0.0,
        "license": info.get("license"),
        "url": info.get("url"),
    })

step7_df = pd.DataFrame(summary_rows)
step7_csv = DATA_ROOT / "corpus_summary_step7.csv"
step7_df.to_csv(step7_csv, index=False, encoding="utf-8-sig")
print("Step 7 Corpus Summary:")
display(step7_df)

# Save global manifests
manifest_app = pd.concat(app_parts, ignore_index=True)
manifest_app.to_csv(DATA_ROOT / "corpus_manifest_app.csv", index=False, encoding="utf-8-sig")

if full_parts:
    manifest_full = pd.concat(full_parts, ignore_index=True)
    if "duration" not in manifest_full.columns:
        manifest_full["duration"] = (pd.to_numeric(manifest_full["end"]) - pd.to_numeric(manifest_full["start"])).round(3)
    manifest_full.to_csv(DATA_ROOT / "corpus_manifest_full.csv", index=False, encoding="utf-8-sig")
    splits = split_corpus(manifest_full, out_dir=DATA_ROOT / "splits", ratios=(0.70, 0.15, 0.15))
    print("Split status:", splits.get("status"))

# Create single final zip archive in /kaggle/working (excludes large intermediate full audio files)
zip_path = WORK_ROOT / "corpus_final_output.zip"
skip_names = {"audio.wav", "audio_conditioned.flac", "video.mp4"}
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(DATA_ROOT.rglob("*")):
        if f.is_file() and f.name not in skip_names and not f.name.endswith(".zip"):
            zf.write(f, arcname=f.relative_to(WORK_ROOT))

print(f"\nFinal Kaggle output archive ready: {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)")
print("All files remain saved on Kaggle under:", DATA_ROOT.resolve())